# LucidPPN reproduction: Stage 4 (experiment) and Stage 5 (deployment) on Colab

Runtime -> Change runtime type -> **T4 GPU**. Run the cells top to bottom.

**This notebook does not change the team's Stage 3 work.** It clones the repository as committed and retrains with the
team's own unmodified `train_quick.py` (Jump 4 settings). The only new code is uploaded as a zip: the Stage 4
evaluation script and the deployment app. Nothing is committed or pushed.

The Stage 3 run kept its checkpoint and part maps only in Colab's temporary storage, so they are gone. This notebook
regenerates them once (about 1 hour) and **saves them to Google Drive**, so later sessions can skip that step.

In [ ]:
# 1. Code: the team's repository as committed on GitHub (read-only use, nothing is pushed)
%cd /content
!git clone https://github.com/rayyansgit/lucidppn-reproduction.git
%cd /content/lucidppn-reproduction
!pip -q install streamlit fastapi uvicorn python-multipart

In [ ]:
# 2. Upload notebooks/stage4_6_files.zip from your computer (Stage 4 + 5 code; adds new files only)
from google.colab import files
uploaded = files.upload()
!unzip -n stage4_6_files.zip   # -n: never overwrite an existing file
!git status --short           # should list only new (??) files, no modified (M) ones

In [ ]:
# 3. Google Drive: keeps the checkpoint, part maps and results between sessions
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/lucidppn'
os.makedirs(DRIVE, exist_ok=True)
HAVE_SAVED = os.path.exists(f'{DRIVE}/checkpoint_quick_train.pt') and os.path.exists(f'{DRIVE}/CUB_subset_maps')
print('Saved checkpoint + maps found on Drive:', HAVE_SAVED)

In [ ]:
# 4. Data: CUB-200-2011 (1.1 GB) -> 25-class bbox-cropped subset (team's prepare_cub_subset.py)
!wget -q -c https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz
!tar -xzf CUB_200_2011.tgz
!python prepare_cub_subset.py

### 5. Part maps + Jump 4 model
On the first run this repeats the team's Stage 3 pipeline with their unmodified scripts:
- PDiscoNet: 5 epochs.
- Part maps.
- MetiNet with the Jump 4 settings: 82 epochs, about 40-60 minutes on a T4.

The log goes to `training_log_JUMP4_rerun.txt`, so the team's original log is not touched. Expect a final accuracy
close to, but not exactly, 81.9%: GPU training is not bit-for-bit reproducible. Report the rerun's number next to the
original in the Stage 4 section.

In [ ]:
if HAVE_SAVED:
    !cp -r $DRIVE/CUB_subset_maps ./CUB_subset_maps
    !cp $DRIVE/checkpoint_quick_train.pt MetiNet/
else:
    !cd part_detection && python train_pdisco_quick.py 2>&1 | tail -3 && python generate_maps_quick.py 2>&1 | tail -2
    !cd MetiNet && python train_quick.py 2>&1 | tee training_log_JUMP4_rerun.txt | grep --line-buffered -E "^Epoch|Done"
    !cp -r CUB_subset_maps $DRIVE/ && cp MetiNet/checkpoint_quick_train.pt MetiNet/training_log_JUMP4_rerun.txt $DRIVE/

## Stage 4: our experiment (A. ablation)

**Question:** how much of our LucidPPN's accuracy comes from the ColorNet branch, and how much robustness to colour
changes does that cost?

`eval_ablation.py` scores the same checkpoint two ways, the full LucidPPN prediction and the grayscale ShapeTexNet
branch on its own, on the clean test set and with the hue randomly rotated (luminance kept) at increasing strength.
This mirrors the paper's Tables 2 and 3. No retraining is needed, because ColorNet multiplies a *detached* copy of the
ShapeTexNet maps, so ShapeTexNet is trained exactly as if ColorNet were absent.

In [ ]:
%cd /content/lucidppn-reproduction/MetiNet
!python eval_ablation.py --checkpoint checkpoint_quick_train.pt --seeds 3
from IPython.display import Image as Show, Markdown
display(Markdown(open('results/ablation.md').read()))
display(Show('results/ablation_hue.png'))
!cp -r results $DRIVE/

### Optional follow-up (B): is the part-alignment loss still needed with the long schedule?
Jumps 2 and 3 compared part maps vs no part maps only at 10 epochs. This re-runs the Jump 4 schedule with
`part_weight=0` (~1 h). It runs on a *copy* of the team's script (`train_quick_noparts.py`); the original is not
edited. Skip it if you're short on GPU quota.

In [ ]:
RUN_PART_ABLATION = False
if RUN_PART_ABLATION:
    !sed -e "s/part_weight=1.0,/part_weight=0.0,/" -e "s/checkpoint_quick_train.pt/checkpoint_no_parts.pt/" -e "s#runs/quick_train#runs/no_parts#" -e "s/WITH part-detection maps (part_weight=1.0)/WITHOUT part maps (part_weight=0)/" train_quick.py > train_quick_noparts.py
    !python train_quick_noparts.py 2>&1 | tee training_log_no_parts.txt | grep --line-buffered -E "^Epoch|Done"
    !python eval_ablation.py --checkpoint checkpoint_no_parts.pt --out_dir results_no_parts
    !cp -r results_no_parts checkpoint_no_parts.pt training_log_no_parts.txt $DRIVE/

## Stage 5: deployment

**API (FastAPI)**: start it in the background and send one test image.

In [ ]:
%cd /content/lucidppn-reproduction
import subprocess, time, glob
api = subprocess.Popen(['uvicorn', 'deploy.api:app', '--port', '8000'])
time.sleep(25)
example = sorted(glob.glob('CUB_subset/test/Cardinal/*.jpg'))[0]
print(example)
!curl -s -F "file=@{example}" "http://localhost:8000/predict?top_k=3" | python -m json.tool
!curl -s -F "file=@{example}" "http://localhost:8000/predict?top_k=3&hue_shift=0.5" | python -m json.tool

**Web app (Streamlit)**: open the printed `trycloudflare.com` link, pick a test image, and move the hue slider.
Take the screenshots / screen recording for the report here. Stop the cell when you're done.

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
app = subprocess.Popen(['streamlit', 'run', 'deploy/app.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(10)
!./cloudflared tunnel --url http://localhost:8501 2>&1 | grep --line-buffered -o 'https://[a-z0-9-]*\.trycloudflare\.com'

## Bring the results back
Download `lucidppn/results/` (and `training_log_JUMP4_rerun.txt`) from Google Drive into the local repository folder,
e.g. as `MetiNet/results/`. Then paste `ablation.md` and `ablation_hue.png` into the TODO slots in `EXPERIMENTS.md` and
`REPORT.md`.